# FracAtlas model comparison

This notebook trains three independent image classifiers on the same FracAtlas split. The candidate used by the FractureCare API is selected on the validation split by **fracture average recall** (the average recall of ONE_FRACTURE and MULTIPLE_FRACTURES), with all-class average F1 as the tie-breaker. The selected model is then evaluated once on the untouched test split.

Why each model is included

Custom CNN - A transparent, lightweight baseline that can be changed easily and establishes a reference point.

MobileNetV2 - An efficient transfer-learning model suited to limited data and practical CPU inference.

EfficientNetB0 - A stronger accuracy/efficiency candidate whose compound scaling often performs well on image classification.


In [1]:
from pathlib import Path
import json
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "1")
import math
import sys
import gc

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

# Allow TensorFlow to grow GPU memory as needed instead of reserving a fixed block.
_available_gpus = tf.config.list_physical_devices('GPU')
for _gpu in _available_gpus:
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except RuntimeError:
        pass
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
SERVICE_DIR = next(path for path in candidates if (path / 'app').is_dir() and (path / 'requirements.txt').exists())
PROJECT_DIR = SERVICE_DIR.parent
sys.path.insert(0, str(SERVICE_DIR))
from app.config import ARTIFACT_DIR, DATASET_CSV, IMAGE_DIR, IMAGE_SIZE, SEED
from app.data import load_manifest
from app.labels import CLASS_NAMES
from app.metrics import SELECTION_METRIC, calculate_classification_metrics, select_best_result
from app.model import MODEL_NAMES, build_model, build_transfer_model

MODEL_EPOCHS = 5
BATCH_SIZE = 8
USE_IMAGENET_WEIGHTS = True
MODEL_DIR = ARTIFACT_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('Dataset:', DATASET_CSV)

I0000 00:00:1788032928.605135     511 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788032930.602218     511 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1788032931.560060     511 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


TensorFlow: 2.21.0
Dataset: /mnt/c/Users/Rushd/OneDrive - wslqd/Documents/Uni Documents/ICBT/Development Project Final Year/Final Documents/fracturecare-prototype/Dataset/FracAtlas/dataset.csv


In [ ]:
frame = load_manifest()
train, holdout = train_test_split(frame, test_size=0.2, random_state=SEED, stratify=frame['label_index'])
validation, test = train_test_split(holdout, test_size=0.5, random_state=SEED, stratify=holdout['label_index'])
train, validation, test = [part.reset_index(drop=True) for part in (train, validation, test)]
print(f'Images: {len(frame):,} | train: {len(train):,} | validation: {len(validation):,} | test: {len(test):,}')
display(frame['label'].value_counts().reindex(CLASS_NAMES).rename('count').to_frame())

W0000 00:00:1788032966.809856     511 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


In [ ]:
def make_dataset(dataframe, shuffle=False):
    paths = dataframe['path'].to_numpy()
    labels = dataframe['label_index'].to_numpy(dtype=np.int32)
    def load(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, IMAGE_SIZE)
        return image, label
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(load, num_parallel_calls=1)
    dataset = dataset.apply(tf.data.experimental.ignore_errors())
    return dataset.batch(BATCH_SIZE).prefetch(1)

train_dataset = make_dataset(train, shuffle=True).repeat()
validation_dataset = make_dataset(validation).repeat()
validation_eval_dataset = make_dataset(validation)
test_dataset = make_dataset(test)
TRAIN_STEPS = math.ceil(len(train) / BATCH_SIZE)
VALIDATION_STEPS = math.ceil(len(validation) / BATCH_SIZE)
weights = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=train['label_index'].to_numpy())
class_weights = {index: float(weight) for index, weight in enumerate(weights)}
print('Class weights:', class_weights)

## Train each model separately

Each loop iteration creates a fresh model, trains it from the same training split, selects its best validation checkpoint, and evaluates it only once on the held-out test split. ImageNet weights are used only by the two transfer-learning models.

In [ ]:
def create_model(model_name):
    if model_name == 'custom_cnn':
        return build_model()
    weights = 'imagenet' if USE_IMAGENET_WEIGHTS else None
    return build_transfer_model(model_name, weights=weights)

model_paths = {}
histories = {}
results = []

for model_name in MODEL_NAMES:
    print(f'\n===== Training {model_name} =====', flush=True)
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = create_model(model_name)
    path = MODEL_DIR / f'{model_name}.keras'
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(path, monitor='val_accuracy', save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6),
    ]

    history = model.fit(train_dataset, validation_data=validation_dataset, epochs=MODEL_EPOCHS, steps_per_epoch=TRAIN_STEPS, validation_steps=VALIDATION_STEPS, class_weight=class_weights, callbacks=callbacks, shuffle=False)
    histories[model_name] = history.history
    model_paths[model_name] = path
    del model
    gc.collect()
    tf.keras.backend.clear_session()

In [ ]:
for model_name, path in model_paths.items():
    model = tf.keras.models.load_model(path)
    probabilities = model.predict(validation_eval_dataset, verbose=0)
    predicted = probabilities.argmax(axis=1)
    actual = np.concatenate([labels.numpy() for _, labels in validation_eval_dataset], axis=0)
    metrics = calculate_classification_metrics(actual, predicted)
    results.append({'model': model_name, **metrics})
    print(f'\n{model_name}')
    print(classification_report(actual, predicted, target_names=CLASS_NAMES, zero_division=0).replace('macro avg', 'average'))

results_df = pd.DataFrame(results).sort_values([SELECTION_METRIC, 'macro_f1', 'balanced_accuracy'], ascending=False).reset_index(drop=True)
display(results_df.style.format({column: '{:.4f}' for column in results_df.columns if column != 'model'}))

## Comparison Charts

In [ ]:
# Export validation comparison charts.
metric_columns = {
    'Accuracy': 'accuracy',
    'Average precision': 'macro_precision',
    'Average recall': 'macro_recall',
    'Average F1': 'macro_f1',
}

metric_frame = results_df.set_index('model')[[column for column in metric_columns.values()]]
metric_frame = metric_frame.rename(columns={value: key for key, value in metric_columns.items()})
ax = metric_frame.plot(kind='bar', figsize=(12, 6), ylim=(0, 1), width=0.78)
ax.set_title('Validation metrics by model')
ax.set_xlabel('Model')
ax.set_ylabel('Score (0–1)')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.25)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=2, fontsize=8)
plt.xticks(rotation=0)
plt.tight_layout()
comparison_plot_path = ARTIFACT_DIR / 'model_metric_comparison.png'
plt.savefig(comparison_plot_path, dpi=220, bbox_inches='tight')
plt.show()

fracture_columns = {
    'Fracture precision': 'fracture_macro_precision',
    'Fracture recall': 'fracture_macro_recall',
    'Fracture F1': 'fracture_macro_f1',
}

fracture_frame = results_df.set_index('model')[[column for column in fracture_columns.values()]]
fracture_frame = fracture_frame.rename(columns={value: key for key, value in fracture_columns.items()})
ax = fracture_frame.plot(kind='bar', figsize=(11, 5.5), ylim=(0, 1), width=0.72)
ax.set_title('Validation metrics for fracture classes only')
ax.set_xlabel('Model')
ax.set_ylabel('Score (0–1)')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.25)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=2, fontsize=8)
plt.xticks(rotation=0)
plt.tight_layout()
fracture_plot_path = ARTIFACT_DIR / 'fracture_metric_comparison.png'
plt.savefig(fracture_plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', comparison_plot_path)
print('Saved:', fracture_plot_path)


In [ ]:
best_result = select_best_result(results)
best_name = best_result['model']
best_path = model_paths[best_name]
best_model = tf.keras.models.load_model(best_path)

final_probabilities = best_model.predict(test_dataset, verbose=0)
final_predicted = final_probabilities.argmax(axis=1)
final_actual = np.concatenate([labels.numpy() for _, labels in test_dataset], axis=0)
final_test_metrics = calculate_classification_metrics(final_actual, final_predicted)

print('Final test-set metrics:')
print(classification_report(final_actual, final_predicted, target_names=CLASS_NAMES, zero_division=0))
best_model.save(ARTIFACT_DIR / 'fracture_classifier.keras')
results_df.to_csv(ARTIFACT_DIR / 'model_comparison.csv', index=False)
test.to_csv(ARTIFACT_DIR / 'test_manifest.csv', index=False)

metadata = {
    'modelVersion': f'fracatlas-{best_name}-1.0.0',
    'selectedModel': best_name,
    'classes': list(CLASS_NAMES),
    'imageSize': list(IMAGE_SIZE),
    'datasetCsv': str(DATASET_CSV),
    'trainCount': int(len(train)),
    'validationCount': int(len(validation)),
    'testCount': int(len(test)),
    'selectionMetric': SELECTION_METRIC,
    'comparison': results_df.to_dict(orient='records'),
    'finalTestMetrics': final_test_metrics,
}

(ARTIFACT_DIR / 'model_metadata.json').write_text(json.dumps(metadata, indent=2, default=float), encoding='utf-8')
print(f'Selected {best_name} using {SELECTION_METRIC} (average F1 tie-breaker) and saved it to {ARTIFACT_DIR / "fracture_classifier.keras"}')
display(results_df)